In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260829_063459"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))
snapshots

,ts,trade_latency,depth_latency,exchange_latency,symbol,mid,mid_tick,microprice,microprice_dev,microprice_error,...,ask_delta,quote_churn,future_mid_100ms,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms
0,1787985301844,0,40,0,PEPEUSDT,0.000004,363,0.000004,-2.925525e-10,2.925525e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
1,1787985301944,0,42,0,PEPEUSDT,0.000004,363,0.000004,-2.925525e-10,2.925525e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
2,1787985304644,0,50,0,PEPEUSDT,0.000004,363,0.000004,-2.925525e-10,2.925525e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
3,1787985305444,0,39,0,PEPEUSDT,0.000004,363,0.000004,-2.809600e-10,2.809600e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
4,1787985305744,0,40,0,PEPEUSDT,0.000004,363,0.000004,-2.818604e-10,2.818604e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23085,1787994879244,38,39,0,PEPEUSDT,0.000004,362,0.000004,2.557403e-09,-2.557403e-09,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN
23086,1787994879844,38,39,0,PEPEUSDT,0.000004,362,0.000004,2.526454e-09,-2.526454e-09,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN
23087,1787994880944,38,44,0,PEPEUSDT,0.000004,362,0.000004,2.525616e-09,-2.525616e-09,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN
23088,1787994881245,38,40,0,PEPEUSDT,0.000004,362,0.000004,2.525616e-09,-2.525616e-09,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN


In [2]:
"""

Does residual_delta explain the residual error in my reservation price (mid + struct_delta + micro_signal) | market state?

"""

df = snapshots

horizons = [100, 500, 1000, 5000]

df["reservation_y"] = df["mid"] + df["struct_delta"] + df["micro_signal_delta"] # reconstructed reservation, just to be clear what goes in reservation

for h in horizons:
    df[f"y_{h}ms"] = np.log(df[f"future_mid_{h}ms"] / df["reservation_y"]) # residuals with center = mid + struct_delta + micro_drift

feature_cols = [
    # raw microstructure
    "spread",
    "volatility",
    "order_imbalance",
    "trade_imbalance",
    "microprice_dev"
]

df = df.dropna()

split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

X_train = train[feature_cols].to_numpy(dtype=np.float32)
X_test = test[feature_cols].to_numpy(dtype=np.float32)

In [4]:
def ic(pred, y):
    return np.corrcoef(pred, y)[0, 1]

def rank_ic(pred, y):
    return spearmanr(pred, y).statistic

results = {}
models = {}

for h in horizons:

    y_train = train[f"y_{h}ms"]
    y_test = test[f"y_{h}ms"]

    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)

    pred = model.predict(X_test)
    true = y_test.values

    # -------------------------
    # signal quality
    # -------------------------
    residual_ic = ic(pred, true)
    residual_rank_ic = rank_ic(pred, true)

    # -------------------------
    # direction accuracy
    # -------------------------
    hit_rate = (np.sign(pred) == np.sign(true)).mean()

    # -------------------------
    # true economic interpretation
    # -------------------------
    pnl_proxy = np.mean(pred * true)
    pnl_std = np.std(pred * true) + 1e-9
    sharpe_proxy = pnl_proxy / pnl_std

    models[h] = {
        "model": model
    }
    results[h] = {
        "Residual_IC": residual_ic,
        "Residual_Rank_IC": residual_rank_ic,
        "HitRate": hit_rate,
        "PnLProxy": pnl_proxy,
        "SharpeProxy": sharpe_proxy,
    }

pd.set_option('display.float_format', '{:.18f}'.format)
results_df = pd.DataFrame(results)
results_df

,100,500,1000,5000
Residual_IC,0.212718864423984022,0.246006956902442592,0.360878825504845790,0.308655042072307484
Residual_Rank_IC,0.118048346829304304,0.131365842082407730,0.165120942038248758,0.287958397094537966
HitRate,0.003466204506065858,0.009098786828422877,0.015597920277296361,0.051126516464471403
PnLProxy,0.000000001261032401,0.000000009822067910,0.000000032767299907,0.000000080762409109
SharpeProxy,0.038236640336163787,0.059338133961832393,0.081330600832840202,0.156438412048819198


In [ ]:
# artifact = {
#     "model": models[1000]["model"],
#     "feature_cols": feature_cols,
#     "target": "log(future_mid/(mid + struct_delta + micro_signal_delta))",
#     "horizon_ms": 1000,
# }

# joblib.dump(artifact, "data/residual_model_3.pkl")

['data/residual_model_3.pkl']

In [5]:
"""
Your target is
y = np.log(future_mid / reservation_y) -> expected log return

where
reservation_y = mid + struct_delta + micro_signal_delta

So the model learns

y^ = log(future_mid / reservation).

Equivalently,
future_mid = reservation ⋅ e^y^.

What your inference does

You compute
double reservation = features.mid + struct_delta + micro_signal_delta;
double expected_return = residual_model->predict(features);
double residual_center = reservation * exp(expected_return * effective_k);

If effective_k == 1, then
residual_center = reservation * exp(expected_return);

which is exactly
future_mid = reservation ⋅ e^y^.

Then return residual_center - reservation;
is residual Δ = future_mid - reservation,

which is the residual price adjustment you want.
"""

def export_xgb(model_name, target, horizon_ms):
    model = models[horizon_ms]["model"]
    model.save_model(f"data/{model_name}_xgb.json")

    artifact = {
        "model_name": f"{model_name}",
        "target": target,
        "model_file": f"data/{model_name}_xgb.json",
        "horizon_ms": horizon_ms,
        "feature_cols": feature_cols,
        "feature_dim": len(feature_cols)
    }

    with open(f"data/{model_name}.json", "w") as f:
        json.dump(artifact, f, indent=4)
    
    print(f"['data/{model_name}.json']")
    print(f"['data/{model_name}_xgb.json']")

export_xgb(model_name="residual_model_pepe", target="log(future_mid/(mid + struct_delta + micro_signal_delta))", horizon_ms=1000)

['data/residual_model_pepe.json']
['data/residual_model_pepe_xgb.json']
